# 03 - Model 1: LightGBM

TF-IDF -> TruncatedSVD (dense text signal) + one-hot categoricals + the hand-crafted
features from `src/features.py`, fed to LightGBM with `scale_pos_weight` for the
class imbalance.

This is the model that can use *both* halves of the data: the wording of the advert
and the metadata around it (no logo, no salary, vague location).

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether this runs from notebooks/ or the project root.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config, evaluation, features, models, preprocessing

config.set_seed()
config.ensure_dirs()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
train, val, test = preprocessing.load_splits()

# Fit every transformer on train only, then reuse them - no refitting on val/test.
X_train, y_train, artifacts = preprocessing.preprocess_data(train)
X_val, y_val = preprocessing.preprocess_new_data(val, artifacts)
X_test, y_test = preprocessing.preprocess_new_data(test, artifacts)

print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")
print(f"{config.SVD_COMPONENTS} SVD text components + "
      f"{X_train.shape[1] - config.SVD_COMPONENTS - len(features.NUMERIC_FEATURES)} one-hot + "
      f"{len(features.NUMERIC_FEATURES)} hand-crafted")
X_train.iloc[:3, -8:]

In [ ]:
scale_pos_weight = models.compute_scale_pos_weight(y_train)
print(f"scale_pos_weight = {scale_pos_weight:.1f} (negatives per positive)")

model = models.train_lightgbm(X_train, y_train, X_val, y_val)
print(f"stopped at {model.best_iteration_ or model.n_estimators} trees "
      f"(early stopping on validation average precision)")

### Evaluate train and validation separately

In [ ]:
proba_train = models.predict_proba(model, X_train)
proba_val = models.predict_proba(model, X_val)

train_metrics = evaluation.evaluate_predictions(y_train, proba_train)
val_metrics = evaluation.evaluate_predictions(y_val, proba_val)

print("train:", {k: round(v, 4) for k, v in train_metrics.items()})
print("val:  ", {k: round(v, 4) for k, v in val_metrics.items()})
print(f"\nAP gap (train - val): "
      f"{train_metrics['average_precision'] - val_metrics['average_precision']:.3f}")

### Tune the threshold on validation, then freeze it

In [ ]:
best_threshold, best_f1 = evaluation.tune_threshold(y_val, proba_val, beta=1.0)
print(f"best F1 threshold: {best_threshold:.3f} (F1={best_f1:.3f})")

# beta=2 weights recall higher - the right call if a missed scam costs more than a
# false alarm. Reported for context; the frozen threshold below is the F1 one.
recall_threshold, recall_f2 = evaluation.tune_threshold(y_val, proba_val, beta=2.0)
print(f"best F2 threshold: {recall_threshold:.3f} (F2={recall_f2:.3f})")

precision_threshold, recall_at_90 = evaluation.threshold_for_precision(y_val, proba_val, 0.9)
print(f"90% precision at threshold {precision_threshold:.3f} -> catches {recall_at_90:.1%} of scams")

val_tuned = evaluation.evaluate_predictions(y_val, proba_val, threshold=best_threshold)
evaluation.log_experiment("lightgbm_tfidf_svd_meta", val_tuned, split="val",
                          notes=f"SVD={config.SVD_COMPONENTS}, spw={scale_pos_weight:.1f}, F1-tuned")
print("\nval @tuned:", {k: round(v, 3) for k, v in val_tuned.items()})

In [ ]:
proba_test = models.predict_proba(model, X_test)
test_metrics = evaluation.evaluate_predictions(y_test, proba_test, threshold=best_threshold)
print("test:", {k: round(v, 4) for k, v in test_metrics.items()})

evaluation.log_experiment("lightgbm_tfidf_svd_meta", test_metrics, split="test",
                          notes="threshold frozen from val")
models.save_test_predictions("lightgbm_tfidf_svd_meta", y_test, proba_test)

## Curves and confusion matrix

In [ ]:
evaluation.plot_pr_curve(y_test, proba_test, label="LightGBM", save_as="10_lgbm_pr_curve.png")
plt.show()
evaluation.plot_roc_curve(y_test, proba_test, label="LightGBM", save_as="11_lgbm_roc_curve.png")
plt.show()
evaluation.plot_confusion_matrix(y_test, (proba_test >= best_threshold).astype(int),
                                 title=f"LightGBM (test, t={best_threshold:.2f})",
                                 save_as="12_lgbm_confusion_matrix.png")
plt.show()

## Feature importance

`svd_*` are compressed text directions and are not individually interpretable - what
matters is how high the *hand-crafted* features rank against them.

In [ ]:
importance = models.feature_importance_frame(model)
top = importance.head(25).sort_values("gain")

fig, ax = plt.subplots(figsize=(8, 7))
colours = ["#c44e52" if not f.startswith("svd_") else "#a6b7d1" for f in top["feature"]]
ax.barh(top["feature"], top["gain"], color=colours)
ax.set_xlabel("gain")
ax.set_title("Top 25 features (red = hand-crafted / metadata, grey = SVD text)")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "13_lgbm_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

engineered = importance[~importance["feature"].str.startswith("svd_")]
print("Best non-text features:")
print(engineered.head(12).to_string(index=False))

### SHAP (optional)

In [ ]:
try:
    import shap
except ImportError:
    print("shap not installed - skipping (pip install shap). Gain importances above cover the basics.")
else:
    sample = X_val.sample(n=min(1000, len(X_val)), random_state=config.SEED)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)
    # Binary LightGBM may return one array or a list of two; take the positive class.
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    shap.summary_plot(shap_values, sample, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / "14_lgbm_shap_summary.png", dpi=150, bbox_inches="tight")
    plt.show()

## Error analysis: which scams slip through?

False negatives are the expensive mistakes. Read a few and look for the pattern -
typically they are the well-written fakes that filled in every field.

In [ ]:
# top_n=len(test) returns every false negative; the print loop shows the worst few.
false_negatives = evaluation.get_errors(test, y_test, proba_test, best_threshold,
                                        kind="false_negatives", top_n=len(test))
print(f"{len(false_negatives)} scams missed out of {int(y_test.sum())} "
      f"({len(false_negatives) / max(int(y_test.sum()), 1):.1%} of them)\n")

for _, row in false_negatives.head(5).iterrows():
    print(f"--- p(fraud)={row['predicted_proba']:.3f} | {row['title'][:70]}")
    print(f"    logo={row['has_company_logo']} questions={row['has_questions']} "
          f"profile_missing={row['company_profile_is_missing']} "
          f"words={row['full_text_word_count']}")
    print(f"    {row['description'][:220]}...\n")

In [ ]:
# Compare every missed scam against every caught one - averaged over 5 rows this
# would be noise.
profile_cols = ["has_company_logo", "has_questions", "company_profile_is_missing",
                "full_text_word_count", "caps_ratio"]
caught = test.loc[(y_test == 1) & (proba_test >= best_threshold)]

comparison = pd.DataFrame({
    "missed_scams": false_negatives[profile_cols].mean(),
    "caught_scams": caught[profile_cols].mean(),
    "real_postings": test.loc[y_test == 0, profile_cols].mean(),
})
print("Why the missed scams blend in:")
print(comparison.round(3).to_string())

### False positives - the real postings we would annoy

In [ ]:
false_positives = evaluation.get_errors(test, y_test, proba_test, best_threshold,
                                        kind="false_positives", top_n=3)
for _, row in false_positives.iterrows():
    print(f"--- p(fraud)={row['predicted_proba']:.3f} | {row['title'][:70]}")
    print(f"    logo={row['has_company_logo']} profile_missing={row['company_profile_is_missing']}")
    print(f"    {row['description'][:200]}...\n")

## Save the model and the fitted transformers

In [ ]:
models.save_model(model, "lightgbm_tfidf_svd_meta")
# The artifacts must travel with the model - without them new data cannot be
# transformed the same way.
models.save_model(artifacts, "lightgbm_artifacts")
print(f"\nfrozen decision threshold: {best_threshold:.4f}")
evaluation.load_experiments()

Next: `04_sentence_transformers.ipynb`.